# Aggregating Point Data with a Regular Grid


In the previous section, we analysed data within administrative boundaries. However, administrative units are not always well suited to spatial analysis — they can vary considerably in size and shape.

A common alternative is a **regular grid** — dividing the study area into equal-sized cells. This allows for more uniform spatial analysis and makes it easier to detect patterns in the distribution of features.

In this section, we will map café density in the Innere Stadt, the first district of Vienna. To do this, we will:

- Build a regular grid of square cells covering the study area.
- Perform a spatial join to count how many cafés fall within each cell.
- Calculate café density (count per km²) per cell and visualise the result.


## 0. Importing Libraries and Preparing the Data


### 0.1. Importing Libraries


In [ ]:
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt

from shapely.geometry import box

# keep all downloaded OSM responses in one cache folder at the root of the repository
ox.settings.cache_folder = "../../cache"

### 0.2. Preparing the Data


We start with the boundary of the Innere Stadt, loaded from OpenStreetMap.


In [ ]:
area_name = "Innere Stadt, Vienna, Austria"
admin_border = ox.geocode_to_gdf(area_name)

admin_border.explore(tiles="cartodbpositron")

We also need café locations from OpenStreetMap, with only the relevant attributes kept.


In [ ]:
tags = {
    "amenity": "cafe"
}

cafes = ox.features_from_place(
    area_name,
    tags
)

Keep only the useful columns.


In [ ]:
cafes["osm_id"] = cafes.index.get_level_values("id")
cafes = cafes[["osm_id", "cuisine", "name", "geometry"]]
cafes.head()

And the cafés on a map.


In [ ]:
cafes.explore(tiles="cartodbpositron")

When point features are displayed on a map, it can be difficult to assess their distribution quantitatively — especially in dense areas where points overlap.

For a clearer analysis, we will build a regular grid and aggregate the data by counting how many cafés fall within each cell.


## 1. Building a Regular Grid


Before building the grid, we need to define **the extent it will cover** and choose the **cell size and shape**.

The grid extent is typically set to match the study area (in our case, the district boundary), and the cell size is specified in metres.

Cells can take various shapes — squares or hexagons, for example — but in this section we will use a **square grid** as the simplest and most common option.

This also means we need to pay attention to the CRS: building a grid with metre-based cell sizes requires a **projected coordinate system**, not a geographic one.


### 1.1. Reprojecting the Data


Check the CRS of both datasets, and reproject into the appropriate UTM zone if needed.


In [ ]:
print(f"CRS admin_border: {admin_border.crs}")
print(f"CRS cafes: {cafes.crs}")

EPSG:4326 is the WGS 84 geographic CRS, where coordinates are expressed in degrees — not suitable for distance or area measurements.
Since we need to work with cell sizes in metres, we will reproject the data into a projected coordinate system.


The appropriate UTM zone comes next. All the datasets cover the same area, so this is worked out once.


In [ ]:
utm_crs = admin_border.estimate_utm_crs()

Now let's reproject both datasets:


In [ ]:
admin_border_utm = admin_border.to_crs(utm_crs)
cafes_utm = cafes.to_crs(utm_crs)

### 1.2. Defining the Extent

Next, we need to determine the spatial extent over which the grid will be built. We use the bounding box of the study area for this.

The bounding box is defined by the minimum and maximum coordinate values along the X and Y axes.

In GeoPandas, this is retrieved using the `total_bounds` attribute, which returns four values: `minx`, `miny`, `maxx`, `maxy`.


In [ ]:
minx, miny, maxx, maxy = admin_border_utm.total_bounds

print(f"minx: {minx}, miny: {miny}, maxx: {maxx}, maxy: {maxy}")

These values define the area within which the grid will be constructed.


### 1.3. Setting the Cell Size

Cell size is one of the key parameters when building a regular grid, as it directly affects the results of the analysis.

Cells that are too large over-generalise the data: local variation is smoothed out and fine-grained patterns in feature distribution may be lost. Cells that are too small produce more detail, but can introduce noise and result in many empty cells.

The optimal cell size depends on the specific task and the density of the data, so in practice it is often worth **experimenting with different sizes** and comparing the results.

Cell size has to suit the study area, not just the question. The Innere Stadt measures roughly 2.2 by 2.1 kilometres, so 500-metre cells would give a grid of about 25 — too coarse for any pattern to show. At 250 metres we get around 80 cells: enough for the distribution to become visible, without making the cells so small that most of them come back empty.

Let's define the cell size in metres and store it in the `square_size` variable.


In [ ]:
square_size = 250

### 1.4. Generating the Cells

Let's create the regular grid within the defined extent, using the chosen cell size. The result will be a set of polygons that uniformly tile the study area and can be used for data aggregation.


In [ ]:
# List to store grid cells
grid_cells = []

# Starting coordinates
x0, y0 = minx, miny
x, y = x0, y0

# Build the grid by iterating across the bounding box and creating square cells
while y < maxy:
    while x < maxx:
        cell = box(x, y, x + square_size, y + square_size)
        grid_cells.append(cell)
        x += square_size
    x = x0
    y += square_size

The result:


In [ ]:
# Display the first 10 cells
first_cells = grid_cells[:10]

ax = gpd.GeoSeries(first_cells, crs=utm_crs).plot(
    figsize=(6, 6), edgecolor="blue", facecolor="none", zorder=3
)

# District boundary
admin_border_utm.boundary.plot(ax=ax, color="green", linewidth=1)

plt.title("First grid cells and district boundary")
plt.show()

Note that the grid is built over the bounding box, so some cells may extend beyond the district boundary. These can be filtered out later if needed.


### 1.5. Converting the Grid to a GeoDataFrame

Once the cells have been generated as a list of geometries, we need to convert them to a `GeoDataFrame` to enable further spatial operations. We store this step-by-step version as `grid_manual`; in the next subsection we wrap the same logic into a function and work with its result from there on.

In [ ]:
# Assemble everything into a GeoDataFrame
grid_manual = gpd.GeoDataFrame({"geometry": grid_cells}, crs=utm_crs)

grid_manual.explore(tiles="cartodbpositron")

### 1.6. Grid Generation Function

Let's wrap the logic above into a reusable function that creates a regular grid. It takes a geodataset (to define the extent) and a cell size as inputs.

The function also includes a CRS check: if the data is in a geographic coordinate system, it is automatically reprojected into the appropriate UTM zone.


In [ ]:
def create_regular_grid(data, cell_size):

    # CRS check
    if data.crs is None:
        raise ValueError("Input data has no CRS defined.")

    if data.crs.is_geographic:
        data = data.to_crs(data.estimate_utm_crs())

    # Bounding box
    minx, miny, maxx, maxy = data.total_bounds

    grid_cells = []
    cell_ids = []

    x = minx
    cell_id = 0  # cell counter

    while x < maxx:
        y = miny
        while y < maxy:
            grid_cells.append(
                box(x, y, x + cell_size, y + cell_size)
            )
            cell_ids.append(cell_id)
            cell_id += 1
            y += cell_size
        x += cell_size

    # GeoDataFrame
    grid = gpd.GeoDataFrame(
        {
            "cell_id": cell_ids,
            "geometry": grid_cells
        },
        crs=data.crs
    )

    return grid

Apply it to our data and look at the grid it returns.


In [ ]:
grid = create_regular_grid(admin_border, square_size)

grid.explore(tiles="cartodbpositron")

The grid has been created successfully and is ready for further analysis. From here on we use `grid` — the result of the function — while `grid_manual` above simply shows what the function does internally.

## 2. Counting Points per Polygon

With the regular grid in place, we can now analyse the distribution of point features. We will determine how many points fall within each grid cell using a spatial join, then aggregate the results.


### 2.1. Spatial Join

To assign each café to a grid cell, we perform a spatial join between the point layer (cafés) and the polygon layer (grid cells).

We use the `within` predicate, since we need to find **which polygon each point falls inside**. The join appends the grid cell attributes to each café point.

The grid was built in the UTM CRS estimated inside the function, and the cafés were reprojected into the same zone earlier — let's confirm before joining:

In [ ]:
grid.crs == cafes_utm.crs

Both layers are in the same CRS, so we can perform the join:

In [ ]:
# Spatial join: assign each café to a grid cell
cafes_in_grid = gpd.sjoin(
    cafes_utm,
    grid,
    how="left",
    predicate="within"
)

The result:


In [ ]:
cafes_in_grid[["name", "cell_id"]].head()

We now know which grid cell each café belongs to, and can count the number of cafés in each cell.


### 2.2. Aggregating the Data

Count the points that fall within each grid cell.


We group by cell ID (`cell_id`) and count the number of cafés in each:


In [ ]:
cafe_counts = (
    cafes_in_grid
    .groupby("cell_id")
    .size()
    .reset_index(name="cafe_count")
)

cafe_counts.head()

After aggregation, we have a table with the café count per cell — but without geometry. To proceed with spatial analysis and visualisation, we need to join these counts back to the grid.


### 2.3. Joining the Counts Back to the Grid


We join the aggregated counts back to the grid using a tabular join on `cell_id`, so that each cell contains the number of cafés within it.


In [ ]:
# Merge the grid with the café counts
grid_with_counts = grid.merge(
    cafe_counts,
    on="cell_id",
    how="left"
)

# Fill NaN values (cells with no cafés)
grid_with_counts["cafe_count"] = grid_with_counts["cafe_count"].fillna(0)

grid_with_counts.head()

### 2.4. Calculating Density

Now that we have the café count per cell, we can calculate density by dividing the count by the cell area.

Since the grid is in a projected CRS, cell areas can be correctly computed in square metres. For easier interpretation, we will express density as the number of cafés per square kilometre.

Let's compute the cell area, convert it to square kilometres, and calculate the density:


In [ ]:
# Cell area in square metres
grid_with_counts["cell_area_m2"] = grid_with_counts.geometry.area

# Convert to square kilometres
grid_with_counts["cell_area_km2"] = grid_with_counts["cell_area_m2"] / 1_000_000

# Café density: count per km²
grid_with_counts["cafe_density"] = (
    grid_with_counts["cafe_count"] / grid_with_counts["cell_area_km2"]
)

The result:


In [ ]:
grid_with_counts_nonzero = grid_with_counts[grid_with_counts["cafe_density"] > 0]

grid_with_counts_nonzero.explore(
    column="cafe_density",
    cmap="OrRd",
    legend=True,
    tiles="cartodbpositron"
)

The map shows that cafés in the Innere Stadt are distributed unevenly. This visualisation makes the spatial pattern far clearer than simply plotting individual points.


At this stage we have produced a map that encodes values through a colour scale — this type of visualisation is called a **choropleth map**.

One important rule for choropleth maps: **avoid using absolute counts**. Because a choropleth ties values to the area of each unit, absolute numbers can be misleading when units differ in size.

In our case all cells have the same area, so using absolute counts would not distort the result. However, this is generally considered poor practice, so **relative measures** — such as density (e.g. count per unit area) — are preferred, as they allow more meaningful comparisons across different parts of the study area.


## Summary

In this section, we looked at how to aggregate point data using a regular grid.

We learned how to build a grid of a specified cell size, assign point features to grid cells using a spatial join, and aggregate the data by counting the number of features per cell.

We also calculated feature density and visualised the result as a choropleth map.

Using a regular grid in this way allows us to move from analysing individual points to identifying spatial patterns across the study area.
